# Environment Setup

Download needed Libraries

In [2]:
!pip install torch torchvision scikit-learn numpy matplotlib

Import the libraries

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torchvision.models as models
import numpy as np
import os
from google.colab import drive
print("GPU is available:", torch.cuda.is_available())

GPU is available: True


Connect to Drive

*necessary to save embeddings while the model trains*

In [3]:
drive.mount('/content/drive')
SAVE_PATH = '/content/drive/MyDrive/simclr_model_cw2.pth'
EMBEDDINGS_PATH = '/content/drive/MyDrive/embeddings_cw2.npy'
LABELS_PATH = '/content/drive/MyDrive/labels_cw2.npy'
print("Drive mounted successfully")

Mounted at /content/drive
Drive mounted successfully


# Embeddings

SimCLR Augmentation and Dataset

In [4]:
# does the simclr augmentations as described in the paper
def get_augmented_data():
    return transforms.Compose([
        transforms.RandomResizedCrop(32),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
        transforms.RandomGrayscale(p=0.2), #20% chance of being grayscale
        transforms.ToTensor(),
        transforms.Normalize(
            (0.4914, 0.4822, 0.4465),
            (0.2023, 0.1994, 0.2010)
        )
    ])

# dataset class that returns 2 views of each image
class SimCLRDataset(torch.utils.data.Dataset):
    def __init__(self, transform):
        # downloads the CIFAR images
        self.dataset = torchvision.datasets.CIFAR10(root='./data',train=True,download=True,transform=None)
        self.transform = transform

    def __len__(self):
        return len(self.dataset) # 50,000 images in the dataset

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        view1 = self.transform(image)
        view2 = self.transform(image)
        # view1 and view2 different cause augmentation is randomized
        return view1, view2

simclr_dataset = SimCLRDataset(get_augmented_data()) # doesn't actually do any data processing yet - just configuration
# batch size as provided in research paper
simclr_loader = DataLoader(simclr_dataset,batch_size=512,shuffle=True,drop_last=True)

print(f"Size of dataset: {len(simclr_dataset)}")
print(f"Number of batches: {len(simclr_loader)}")

100%|██████████| 170M/170M [00:13<00:00, 12.5MB/s]


Size of dataset: 50000
Number of batches: 97


SimCLR Model Creation

*tested with dummy images to verify before actual training*

In [5]:
#just builds the structure of the model
class SimCLR(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=None) # without any pre-trained weights
        # resnet.children() returns the layers like convolutional layers, relu, etc.
        # last layer removed because we want the output of the second last layer - 512 dimensional output
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        # gets the 512 dim down to 128 as per the paper
        self.projection = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 128)
        )

    def forward(self, x):
        h = self.backbone(x)
        h = h.view(h.size(0), -1) # flattens the previous output
        z = self.projection(h) # 128 dimensional output
        return h, z

# testing with a dummy to avoid future errors before the actual training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"CPU or GPU being used: {device}")
test_model = SimCLR().to(device)
dummy = torch.randn(2, 3, 32, 32).to(device)
h, z = test_model(dummy)
print(f"Intermediate flattened shape (h): {h.shape}")   # should be (2, 512)
print(f"Final shape (z): {z.shape}") # should be (2, 128)

#free's gpu space
del test_model, dummy

CPU or GPU being used: cuda
Intermediate flattened shape (h): torch.Size([2, 512])
Final shape (z): torch.Size([2, 128])


Contrastive Loss Function

*compares similar and disimilar pairs*

In [6]:
def contrastive_loss(z1, z2, temperature=0.5): # 2 views as embeddings and temperature given by paper
    batch_size = z1.shape[0]
    z1 = F.normalize(z1, dim=1) # l2 normalizing views to ensure only direction taken into consideration not magnitude
    z2 = F.normalize(z2, dim=1)
    z = torch.cat([z1, z2], dim=0) # matrix so each pair can be compared
    # torch.mm does matrix multiplication with its transpose
    # computes a similarity score between each possible embedding
    sim = torch.mm(z, z.T) / temperature
    labels = torch.arange(batch_size, device=z1.device)
    labels = torch.cat([labels + batch_size, labels])
    mask = torch.eye(2 * batch_size, device=z1.device).bool() # True false matrix created
    sim.masked_fill_(mask, float('-inf')) # to ensure model doesn't consider its own pair as most similar
    loss = F.cross_entropy(sim, labels) # cross entropy loss
    return loss

# testing above with fake inputs
z1_test = torch.randn(4, 128).to(device)
z2_test = torch.randn(4, 128).to(device)
test_loss = contrastive_loss(z1_test, z2_test)
print(f"Loss function verified. Test loss: {test_loss.item():.4f}")

Loss function verified. Test loss: 1.9781


Training the SimCLR model

In [9]:
EPOCHS = 50

# checks if model already exists
if os.path.exists(SAVE_PATH):
    print("model already saved - no training required")
    model = SimCLR().to(device)
    model.load_state_dict(torch.load(SAVE_PATH)) # loads the model
else:
    print("training model since none found in drive")
    model = SimCLR().to(device)
    # hyperparameters as mentioned in the paper
    optimizer = torch.optim.SGD(model.parameters(),lr=0.4,momentum=0.9,weight_decay=0.0001)
    # gradually reduceing learning rate from the paper
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS)

    for epoch in range(EPOCHS): # runs number of times of the epoch
        model.train()
        total_loss = 0
        for view1, view2 in simclr_loader: # runs through each image batch (of 512 images each)
            view1, view2 = view1.to(device), view2.to(device)
            _, z1 = model(view1) # only final view required
            _, z2 = model(view2) # not the intermediate flattened one
            loss = contrastive_loss(z1, z2)
            # standard training process
            optimizer.zero_grad() # gradient removed
            loss.backward() # new ones computed
            optimizer.step() # weights updated
            total_loss += loss.item()
        scheduler.step() # lr updated
        avg_loss = total_loss / len(simclr_loader) # loss should be reducing
        print(f"epoch {epoch+1}/{EPOCHS} - loss: {avg_loss:.4f}")

        # saves every 10 epochs
        if (epoch + 1) % 10 == 0:
            torch.save(model.state_dict(), SAVE_PATH)
            print(f"checkpoint save at epoch {epoch+1}")

    print("training done")
    torch.save(model.state_dict(), SAVE_PATH)
    print("model saved to drive")

training model since none found in drive
epoch 1/50 - loss: 6.3361
epoch 2/50 - loss: 6.1285
epoch 3/50 - loss: 6.0604
epoch 4/50 - loss: 6.0165
epoch 5/50 - loss: 5.9732
epoch 6/50 - loss: 5.9426
epoch 7/50 - loss: 5.9256
epoch 8/50 - loss: 5.8927
epoch 9/50 - loss: 5.8627
epoch 10/50 - loss: 5.8426
checkpoint save at epoch 10
epoch 11/50 - loss: 5.8368
epoch 12/50 - loss: 5.8212
epoch 13/50 - loss: 5.8089
epoch 14/50 - loss: 5.7913
epoch 15/50 - loss: 5.7790
epoch 16/50 - loss: 5.7699
epoch 17/50 - loss: 5.7614
epoch 18/50 - loss: 5.7505
epoch 19/50 - loss: 5.7478
epoch 20/50 - loss: 5.7421
checkpoint save at epoch 20
epoch 21/50 - loss: 5.7334
epoch 22/50 - loss: 5.7291
epoch 23/50 - loss: 5.7166
epoch 24/50 - loss: 5.7111
epoch 25/50 - loss: 5.7066
epoch 26/50 - loss: 5.7057
epoch 27/50 - loss: 5.6972
epoch 28/50 - loss: 5.6916
epoch 29/50 - loss: 5.6874
epoch 30/50 - loss: 5.6790
checkpoint save at epoch 30
epoch 31/50 - loss: 5.6782
epoch 32/50 - loss: 5.6684
epoch 33/50 - loss: 